In [1]:
!pip install transformers

In [3]:
import pandas as pd
from collections import Counter

# Load data
df = pd.read_csv(r"/Users/vijay/Desktop/NLP implementation/df_advanced.csv")

# Split labels
df["labels"] = df["labels"].apply(lambda x: x.split("|"))

# Count label frequency
all_labels = [label for sublist in df["labels"] for label in sublist]
label_counts = Counter(all_labels)

# Get top 10 labels
top_labels = [label for label, _ in label_counts.most_common(10)]

print(top_labels)

['earn', 'acq', 'money-fx', 'crude', 'grain', 'trade', 'interest', 'wheat', 'ship', 'corn']


In [4]:
def filter_labels(label_list):
    return [l for l in label_list if l in top_labels]

df["labels"] = df["labels"].apply(filter_labels)

# Remove rows with no labels left
df = df[df["labels"].map(len) > 0]

In [5]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["labels"])

label_names = mlb.classes_
num_labels = len(label_names)

print(num_labels)  # should be 10

10


In [6]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"], y, test_size=0.2, random_state=42
)

In [7]:
from transformers import AutoTokenizer

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)

In [8]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
for name, param in model.named_parameters():
    if "classifier" not in name and "layer.10" not in name and "layer.11" not in name:
        param.requires_grad = False

In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./roberta_r10",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    logging_dir="./logs",
    do_eval=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
from sklearn.metrics import f1_score
import torch

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.3).int()

    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.178895
1000,0.059493
1500,0.042440
2000,0.035109
2500,0.032063
3000,0.029006
3500,0.026117
4000,0.027820
4500,0.023107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4675, training_loss=0.04943880519764946, metrics={'train_runtime': 1039.6159, 'train_samples_per_second': 35.975, 'train_steps_per_second': 4.497, 'total_flos': 4920530145484800.0, 'train_loss': 0.04943880519764946, 'epoch': 5.0})

In [19]:
import torch
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def evaluate_model(model, dataset, threshold=0.3, device=None):
    # Device setup
    if device is None:
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    for item in dataset:
        inputs = {
            "input_ids": item["input_ids"].unsqueeze(0).to(device),
            "attention_mask": item["attention_mask"].unsqueeze(0).to(device)
        }

        labels = item["labels"].cpu().numpy()

        with torch.no_grad():
            outputs = model(**inputs)

        probs = torch.sigmoid(outputs.logits)
        preds = (probs > threshold).int().cpu().numpy()[0]

        all_preds.append(preds)
        all_labels.append(labels)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_micro": f1_score(all_labels, all_preds, average="micro", zero_division=0),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
    }

In [20]:
for t in [0.3, 0.4, 0.5, 0.6]:
    results = evaluate_model(model, val_dataset, threshold=t)
    print(f"Threshold {t}: {results}")

Threshold 0.3: {'accuracy': 0.932620320855615, 'f1_micro': 0.9647283126787417, 'f1_macro': 0.9374894140190315}
Threshold 0.4: {'accuracy': 0.9390374331550803, 'f1_micro': 0.9675870348139256, 'f1_macro': 0.9389101275876616}
Threshold 0.5: {'accuracy': 0.9427807486631016, 'f1_micro': 0.968273189634294, 'f1_macro': 0.9405988572083969}
Threshold 0.6: {'accuracy': 0.9449197860962567, 'f1_micro': 0.96923828125, 'f1_macro': 0.9392962818500219}


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)

In [21]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.237712
1000,0.110326
1500,0.076403
2000,0.055897
2500,0.042660
3000,0.036381
3500,0.026669
4000,0.024777
4500,0.019659


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4675, training_loss=0.06820727690018434, metrics={'train_runtime': 1076.1371, 'train_samples_per_second': 34.754, 'train_steps_per_second': 4.344, 'total_flos': 2477493765120000.0, 'train_loss': 0.06820727690018434, 'epoch': 5.0})

In [23]:
for t in [0.3, 0.4, 0.5, 0.6]:
    results = evaluate_model(model, val_dataset, threshold=t)
    print(f"Threshold {t}: {results}")

Threshold 0.3: {'accuracy': 0.9037433155080213, 'f1_micro': 0.9397937155193092, 'f1_macro': 0.8935463967264029}
Threshold 0.4: {'accuracy': 0.9080213903743316, 'f1_micro': 0.9417733752114037, 'f1_macro': 0.8976929312753568}
Threshold 0.5: {'accuracy': 0.9106951871657754, 'f1_micro': 0.9423778264040846, 'f1_macro': 0.8971833988141021}
Threshold 0.6: {'accuracy': 0.9101604278074866, 'f1_micro': 0.9416952474277315, 'f1_macro': 0.896098717568442}


In [24]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModel.from_pretrained(model_name)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
bert_model.to(device)
bert_model.eval()

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [25]:
import numpy as np

def get_embeddings(texts):
    embeddings = []

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = bert_model(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(cls_embedding[0])

    return np.array(embeddings)

In [26]:
X_train = get_embeddings(train_texts)
X_val = get_embeddings(val_texts)


In [27]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

svm_model = OneVsRestClassifier(LinearSVC())
svm_model.fit(X_train, train_labels)

,estimator,LinearSVC()
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [28]:
from sklearn.metrics import f1_score

y_pred = svm_model.predict(X_val)

print("F1 Micro:", f1_score(val_labels, y_pred, average="micro"))
print("F1 Macro:", f1_score(val_labels, y_pred, average="macro"))

F1 Micro: 0.8911368015414258
F1 Macro: 0.8380346280456863
